# 🛰️ ICARUS_WORKER_EO — Earth Observation Worker
## Dedicated Remote Scientific Worker Runtime (Kaggle Account 2)

**Version:** 0.2.0 (Distributed Architecture)  
**Role:** Earth Observation Processing & Specialist Model Host  
**Primary Model Target:** IBM/NASA Prithvi-EO-2.0-300M  
**Interface:** FastAPI over Encrypted ngrok Tunnel  
**Deployment Profile:** Kaggle Account 2 (GPU T4 x2 or P100)

---

### Worker Responsibilities in 4-Account Topology
- Hosts traditional multispectral algorithms: **NDVI, NDWI, NBR, Area with CRS validation**.
- Provides **GeoTIFF ingestion, band mapping, spatial tiling (224x224), and raster reconstruction**.
- Hosts **Prithvi-EO-2.0-300M** with lazy loading, strict 6-band normalization, and VRAM eviction.
- Serves `/health`, `/capabilities`, `/analyze`, and `/unload-model` endpoints to `ICARUS_CORE`.


---
## 1. Environment Detection & Kaggle Secrets

Inspect GPU resources (VRAM availability) and retrieve `NGROK_AUTH_TOKEN` from Kaggle Secrets.


In [ ]:
# ============================================================
# 1. Setup, Environment & Secrets
# ============================================================
import os
import sys
import json
import time
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

def get_secret(key: str, default: str = "") -> str:
    """Fetches secret from Kaggle Secrets or environment variables."""
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            return user_secrets.get_secret(key)
        except Exception:
            pass
    return os.environ.get(key, default)

NGROK_AUTH_TOKEN = get_secret("NGROK_AUTH_TOKEN")
COPERNICUS_CLIENT_ID = get_secret("COPERNICUS_CLIENT_ID")
NASA_EARTHDATA_USER = get_secret("NASA_EARTHDATA_USERNAME")

# Inspect GPU and PyTorch
try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "None"
    VRAM_MB = torch.cuda.get_device_properties(0).total_mem // (1024**2) if CUDA_AVAILABLE else 0
except ImportError:
    CUDA_AVAILABLE = False
    GPU_NAME = "PyTorch not installed"
    VRAM_MB = 0

print("=" * 60)
print("🛰️  ICARUS_WORKER_EO Runtime Initialization")
print("=" * 60)
print(f"  Platform:          {'Kaggle Notebook' if IS_KAGGLE else 'Local / Server'}")
print(f"  Python Version:    {sys.version.split()[0]}")
print(f"  GPU Accelerator:   {GPU_NAME} ({VRAM_MB} MB VRAM)")
print(f"  CUDA Available:    {CUDA_AVAILABLE}")
print(f"  NGROK_AUTH_TOKEN:  {'✅ Configured' if NGROK_AUTH_TOKEN else '⚠️ Missing (Required for public tunnel)'}")
print("=" * 60)

---
## 2. Core Library Imports

Import the shared schemas, EO pipeline, and Prithvi foundation adapter.


In [ ]:
# ============================================================
# 2. Package Imports
# ============================================================
try:
    import icarus
except ImportError:
    # Ensure working directory is in sys.path
    repo_path = Path("/kaggle/working/ICARUS").resolve()
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
    else:
        sys.path.insert(0, str(Path(".").resolve()))

from icarus.schemas import (
    DomainEnum,
    CapabilityStatus,
    TaskStatus,
    WorkerRuntimeStatus,
    ResearchRequest,
    ResearchResponse,
    TaskMetric,
    ExecutionMetadata,
    CapabilityDescriptor,
    WorkerCapabilitiesResponse,
    HealthCheckResponse,
    UnloadModelRequest,
    UnloadModelResponse,
)
from icarus.workers.eo.eo_pipeline import EOPipeline
from icarus.workers.eo.prithvi_adapter import PrithviEOAdapter, PrithviModelStatus
from icarus.workers.eo.service import app

print("✅ EO Worker modules and FastAPI application loaded.")

---
## 3. Traditional Spectral Pipeline & GeoTIFF Tiling

Verify traditional spectral algorithms (NDVI, NDWI, NBR, surface area with CRS checks, and spatial tiling) independently of deep learning foundation models.


In [ ]:
# ============================================================
# 3. Traditional EO Pipeline Verification
# ============================================================
import numpy as np

# 1. Test NDVI math with synthetic arrays
nir_demo = np.array([[0.6, 0.7], [0.8, 0.5]])
red_demo = np.array([[0.2, 0.1], [0.2, 0.3]])
ndvi_calc = EOPipeline.compute_ndvi(nir_demo, red_demo)
print("NDVI Sample Output:
", ndvi_calc)

# 2. Test vegetation distribution statistics
stats = EOPipeline.compute_vegetation_statistics(ndvi_calc)
print("
Vegetation Statistics Summary:")
print(f"  Mean NDVI: {stats['mean']}")
print(f"  Dense Canopy Fraction: {stats['classification_fractions']['dense_canopy_fraction']}")

# 3. Test CRS Guard on Area Calculation
area_deg = EOPipeline.calculate_surface_area_km2(1000, 10.0, crs="EPSG:4326")
print("
Geographic Area Calculation Guard:", area_deg.get("error", "Passed"))

area_utm = EOPipeline.calculate_surface_area_km2(10000, 10.0, crs="EPSG:32643")
print(f"Projected Area Calculation: {area_utm['area_km2']} {area_utm['unit']}")

# 4. Test Spatial Tiling (224x224) & Reconstruction
sample_scene = np.random.uniform(0, 1, (6, 500, 500)).astype(np.float32)
tiles, coords, orig_shape = EOPipeline.tile_raster(sample_scene, tile_size=224, stride=224)
print(f"
Spatial Tiling: Extracted {len(tiles)} patches of shape {tiles[0].shape} from original {orig_shape}")

reconstructed = EOPipeline.reconstruct_raster(tiles, coords, orig_shape, tile_size=224)
recon_err = float(np.mean(np.abs(sample_scene - reconstructed)))
print(f"Reconstruction Mean Absolute Difference: {recon_err:.6f}")

---
## 4. IBM/NASA Prithvi-EO-2.0-300M Adapter

Inspect the verified architectural specifications, test HLS 6-band normalization, and verify lazy loading with GPU VRAM management.


In [ ]:
# ============================================================
# 4. Prithvi EO 2.0 Adapter & VRAM Management
# ============================================================
adapter = PrithviEOAdapter(device="cuda" if CUDA_AVAILABLE else "cpu")
print("Prithvi Status:", adapter.get_status())

# Verify exact 6 HLS band ordering
print("
Required HLS Band Ordering:")
for i, b in enumerate(adapter.REQUIRED_BANDS):
    print(f"  Channel {i}: {b} (Mean: {adapter.HLS_CHANNEL_MEANS[i]}, Std: {adapter.HLS_CHANNEL_STDS[i]})")

# Test tensor normalization: (Batch=1, Channels=6, Time=1, H=224, W=224)
raw_hls = np.random.uniform(500, 3000, (1, 6, 1, 224, 224)).astype(np.float32)
norm_hls = adapter.normalize_hls_channels(raw_hls)
print("
Normalized Tensor Shape:", norm_hls.shape)
print(f"Mean after norm: {np.mean(norm_hls):.4f}, Std: {np.std(norm_hls):.4f}")

# Lazy Load Check
print("
Loading Prithvi Model...")
ok, msg = adapter.load_model()
print(f"Load Status: {ok} -> {msg}")
print(f"VRAM Allocated: {adapter.vram_allocated_mb} MB")

# Extract Latent Tokens
features = adapter.extract_features(norm_hls)
print("
Feature Extraction Output:")
print(f"  Architecture: {features.get('architecture')}")
print(f"  Token Shape:  {features.get('feature_shape')}")
print(f"  Note:         {features.get('scientific_note')}")

# Unload Model
print("
Unloading Prithvi Model to free memory...")
unloaded, un_msg, freed = adapter.unload_model()
print(f"Unload Result: {un_msg} (Freed: ~{freed} MB)")

---
## 5. In-Process HTTP Contract Verification

Run verification requests directly against the FastAPI `app` using `starlette.testclient.TestClient` before starting the public network server.


In [ ]:
# ============================================================
# 5. In-Process Contract Verification
# ============================================================
from starlette.testclient import TestClient

client = TestClient(app)

print("--- [1. GET /health] ---")
resp_health = client.get("/health")
print(f"Status: {resp_health.status_code}")
print(json.dumps(resp_health.json(), indent=2))

print("
--- [2. GET /capabilities] ---")
resp_caps = client.get("/capabilities")
print(f"Status: {resp_caps.status_code}")
print(f"Total Capabilities Advertised: {len(resp_caps.json()['capabilities'])}")

print("
--- [3. POST /analyze (NDVI Calculation)] ---")
req_ndvi = {
    "domain": "earth_observation",
    "task": "eo_vegetation_analysis",
    "research_question": "Compute NDVI for study quadrant",
    "parameters": {
        "nir": [[0.7, 0.8], [0.6, 0.75]],
        "red": [[0.15, 0.2], [0.18, 0.22]]
    }
}
resp_analyze = client.post("/analyze", json=req_ndvi)
print(f"Status: {resp_analyze.status_code}")
print(json.dumps(resp_analyze.json(), indent=2))

print("
--- [4. POST /analyze (Prithvi Foundation Embeddings)] ---")
req_prithvi = {
    "domain": "earth_observation",
    "task": "eo_foundation_embeddings",
    "research_question": "Extract latent representations with Prithvi EO 2.0"
}
resp_prithvi = client.post("/analyze", json=req_prithvi)
print(f"Status: {resp_prithvi.status_code}")
print(json.dumps(resp_prithvi.json(), indent=2))

print("
--- [5. POST /unload-model] ---")
resp_unload = client.post("/unload-model", json={"model_id": "prithvi-eo-2.0-300m"})
print(f"Status: {resp_unload.status_code}")
print(json.dumps(resp_unload.json(), indent=2))

---
## 6. Launch Public FastAPI Server via ngrok

Expose the worker service to the internet using pyngrok.  
The public URL printed below can be registered on Kaggle Account 1 (`ICARUS_CORE.ipynb`).


In [ ]:
# ============================================================
# 6. Start Public ngrok Tunnel & Background Server
# ============================================================
import threading
import uvicorn

PORT = 8000

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

# Start Uvicorn in background thread
server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)
print(f"✅ Local FastAPI worker listening on http://localhost:{PORT}")

# Start ngrok tunnel if token is available
if NGROK_AUTH_TOKEN:
    try:
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        # Open HTTP tunnel
        public_tunnel = ngrok.connect(PORT, "http")
        public_url = public_tunnel.public_url
        print("\n" + "=" * 65)
        print("🌐 ICARUS_WORKER_EO PUBLIC NGROK TUNNEL ONLINE:")
        print(f"   {public_url}")
        print("=" * 65)
        print("\n👉 Copy this URL and register it on Account 1 (ICARUS_CORE):")
        print(f'   orchestrator.register_worker("{public_url}", name="ICARUS_WORKER_EO")')
    except Exception as e:
        print(f"⚠️ ngrok tunnel launch failed: {str(e)}")
else:
    print("\n⚠️ NGROK_AUTH_TOKEN not provided.")
    print("   To connect this worker to ICARUS_CORE in a multi-notebook setup,")
    print("   add NGROK_AUTH_TOKEN to Kaggle Secrets (Add-ons > Secrets).")